# UNGM (UN Global Marketplace) — Notion Upload

https://www.ungm.org/Public/Notice

Fetches live procurement notices from UNGM (same endpoint, filters and quirks as the OppsLink-side `ungm.ipynb` — this notebook re-implements that fetch/filter/parse logic independently, since this repo runs on its own without reading from the OppsLink repo, same convention as your other four sources) and uploads new ones to the unified Notion database.

**Suggested location:** `sources/UNGM/ungm_notion_upload.ipynb` in this repo — let me know if you want a different filename.

**Filters applied:**
- Type of opportunity: Request for EOI, Request for proposal, Request for quotation, Request for pre-qualification
- Active opportunities only
- Goods and services (UNSPSC):
  - 77000000 - Environmental Services
    - 77100000 - Environmental management
    - 77110000 - Environmental protection
    - 77120000 - Pollution tracking and monitoring and rehabilitation
  - 80000000 - Management and Business Professionals and Administrative Services
    - 80100000 - Management advisory services
    - 80110000 - Human resources services
    - 80120000 - Legal services
    - 80150000 - Trade policy and services
  - 81000000 - Engineering and Research and Technology Based Services
    - 81120000 - Economics
    - 81130000 - Statistics
  - 86000000 - Education and Training Services
    - 86100000 - Vocational training
    - 86110000 - Alternative educational systems
    - 86120000 - Educational institutions
    - 86130000 - Specialized educational services
  - 92000000 - National Defense and Public Order and Security and Safety Services
    - 92100000 - Public order and safety
    - 92110000 - Military services and national defense
    - 92120000 - Security and personal safety
  - 93000000 - Politics and Civic Affairs Services
    - 93100000 - Political systems and institutions
    - 93110000 - Socio political conditions
    - 93120000 - International relations
    - 93130000 - Humanitarian aid and relief
    - 93140000 - Community and social services
    - 93150000 - Public administration and finance services
    - 93160000 - Taxation
    - 93170000 - Trade policy and regulation
- Only fetches notices **published in the last 4 days** (`PUBLISHED_LOOKBACK_DAYS`) — without this, our filters match 1,730 total currently-open notices (mostly long-running EOI/pre-qualification calls), which would be slow to fetch and scrape daily for almost no new results. 4 days covers weekend/schedule gaps; CSV dedup below catches any overlap.

**Known gaps, flagged rather than guessed at:**
- **`CPV Codes` is `"Not Available"` for every UNGM row.** UNGM uses UNSPSC, not CPV, and the search-results view doesn't expose per-notice codes — only the detail page does, in a section I haven't confirmed the markup for yet (same situation we hit with the description text on the OppsLink side, before you sent me that HTML). Happy to wire this in properly if you grab that section's HTML the same way.
- **`Value` is always `"Not Disclosed"`** — UNGM notices don't publish a contract value, unlike the EU/UK sources.
- **`Employer Website` is always blank** — no stable per-agency URL available from this source.
- **`Language` is hardcoded to `"English"`**, matching `eu_commission_notion_upload.ipynb`'s convention — UNGM doesn't expose a language field in what we're parsing.
- **New dependency:** `beautifulsoup4`, same as the OppsLink-side notebook. I haven't seen this repo's own GitHub Actions workflow, so I don't know if it needs adding separately here — flagging rather than guessing.

**API notes — confirmed via live browser Network-tab captures, not guessed:**
- Endpoint: `POST /Public/Notice/Search`, the same internal endpoint UNGM's own search page calls.
- `NoticeTypes` takes UNGM's internal enum strings, not the display labels.
- `UNSPSCs` takes UNGM's own internal database row IDs, not real UNSPSC codes — see the comment on `UNSPSC_FILTER_IDS` below before touching that list.
- Pagination does NOT trust "got fewer results than requested" as an end-of-results signal — every captured payload showed a fixed page size regardless of what we asked for, so this only stops on a genuinely empty page.

### Notion credentials

In [1]:
import os

NOTION_TOKEN = os.environ["NOTION_TOKEN"]

DATABASE_ID = '334701e728cb8096a94cebc0985684a2'

headers_notion = {
    "Authorization": "Bearer " + NOTION_TOKEN,
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28",
}


### Fetch, filter & parse (self-contained — same logic as the OppsLink-side notebook)

In [2]:
import requests
import pandas as pd
import json
import html
import os
import time
from datetime import datetime, timedelta, timezone
from dateutil import parser as _dateparser
from bs4 import BeautifulSoup

SEARCH_URL = "https://www.ungm.org/Public/Notice/Search"
NOTICE_URL_TEMPLATE = "https://www.ungm.org/Public/Notice/{}"

UNGM_HEADERS = {
    "Accept": "*/*",
    "Accept-Language": "en-GB,en;q=0.9",
    "Content-Type": "application/json",
    "Origin": "https://www.ungm.org",
    "Referer": "https://www.ungm.org/Public/Notice",
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.6 Safari/605.1.15"
    ),
    "X-Requested-With": "XMLHttpRequest",
}

NOTICE_TYPES = [
    "RequestForEoi",
    "RequestForProposal",
    "RequestForQuotation",
    "RequestForPreQualification",
]

# Without a PublishedFrom filter, this pulls EVERY currently-open notice matching
# our categories - confirmed on 25-Jul-2026 that this is 1,730 results (many are
# long-open EOI/pre-qualification calls, not new). 4 days covers the Fri-to-Mon
# gap in a Mon-Fri run schedule, plus a day of buffer. CSV dedup below still
# catches any overlap.
PUBLISHED_LOOKBACK_DAYS = 4

# These are UNGM's own internal database row IDs for each UNSPSC code —
# NOT the 8-digit UNSPSC codes themselves. Do not "fix" these back to codes
# like 77000000 / 80100000 / etc. — that silently returns zero results.
# Captured from a live Network payload (Notice/Search request) after manually
# selecting the target categories in the UNGM UI picker.
# To add more categories later: open Goods and services > Search codes in the
# UI, tick the new category, hit Search, and pull the new IDs out of that same
# request's UNSPSCs array — or query Shared/Unspsc/Filter directly to look up
# a code's Id without going through the picker.
UNSPSC_FILTER_IDS = [
    107354, 107355, 108540, 107364, 107365, 107366, 107369, 107374, 107375,
    107414, 107415, 107416, 107417, 107430, 107431, 107432, 107434, 107435,
    107436, 107437, 107438, 107439, 107401, 117154,
]


def build_payload(page_index: int = 0, page_size: int = 100) -> dict:
    """Returns the JSON body for POST /Public/Notice/Search."""
    now_utc = datetime.now(timezone.utc)
    today_str = now_utc.strftime("%d-%b-%Y")
    published_from_str = (now_utc - timedelta(days=PUBLISHED_LOOKBACK_DAYS)).strftime("%d-%b-%Y")
    return {
        "PageIndex": page_index,
        "PageSize": page_size,
        "Title": "",
        "Description": "",
        "Reference": "",
        "PublishedFrom": published_from_str,
        "PublishedTo": "",
        "DeadlineFrom": today_str,   # matches "Active opportunities" behaviour
        "DeadlineTo": "",
        "Countries": [],
        "Agencies": [],
        "UNSPSCs": UNSPSC_FILTER_IDS,
        "NoticeTypes": NOTICE_TYPES,
        "TypeOfCompetitions": [],
        "SortField": "Deadline",
        "SortAscending": True,
        "IsActive": True,
        "IsSustainable": False,
        "isPicker": False,
        "NoticeDisplayType": None,
        "NoticeSearchTotalLabelId": "noticeSearchTotal",
    }


def parse_notices(html_text: str) -> list[dict]:
    """Parse the HTML fragment returned by /Public/Notice/Search into dicts."""
    soup = BeautifulSoup(html_text, "html.parser")
    notices = []

    for row in soup.select("div.tableRow.dataRow"):
        notice_id = row.get("data-noticeid")
        if not notice_id:
            continue

        title_el = row.select_one("span.ungm-title")
        deadline_el = row.select_one("div.deadline span")
        row_cells = row.select("div.tableCell")

        # Cell order: 0=options, 1=title, 2=deadline, 3=published,
        #             4=agency, 5=type, 6=reference, 7=country
        try:
            agency = row_cells[4].get_text(strip=True)
            country = row_cells[7].get_text(strip=True)
        except IndexError:
            agency = country = ""

        notices.append({
            "notice_id": notice_id,
            "title": title_el.get_text(strip=True) if title_el else "",
            "deadline_raw": deadline_el.get_text(strip=True) if deadline_el else "",
            "agency": agency,
            "country": country,
            "url": NOTICE_URL_TEMPLATE.format(notice_id),
        })

    return notices


def clean_ungm_deadline(raw: str) -> str:
    """
    UNGM's deadline field looks like '21-May-2026 23:59\r\n            (GMT -5.00)'.
    Strips the timezone suffix and collapses whitespace, leaving something
    dateutil can parse cleanly.
    """
    if not raw:
        return ""
    cleaned = raw.split("(")[0]
    return " ".join(cleaned.split())


def clean_description(description):
    if not description:
        return "Not Disclosed"
    cleaned = html.unescape(description)
    cleaned = cleaned.replace('\r\n', ' ').replace('\n', ' ')
    return ' '.join(cleaned.split()).strip()


def extract_description(soup: BeautifulSoup) -> str:
    """
    Pulls the free-text description out of a notice detail page.
    Matches on the '.title' text ('Description') rather than a section-specific
    class, since UNGM reuses the same 'ungm-list-item' wrapper for every section
    (Description, Documents, Contacts, UNSPSC codes) — confirmed against real
    markup pulled from a live notice page.
    """
    for item in soup.select("div.ungm-list-item"):
        title_el = item.find("div", class_="title")
        if not title_el or title_el.get_text(strip=True) != "Description":
            continue
        content_divs = item.find_all("div", recursive=False)
        if len(content_divs) < 2:
            continue
        paragraphs = [p.get_text(" ", strip=True) for p in content_divs[1].find_all("p")]
        paragraphs = [p for p in paragraphs if p]
        if paragraphs:
            return "\n".join(paragraphs)
        return content_divs[1].get_text(" ", strip=True)
    return ""


def fetch_notice_description(session: requests.Session, notice_id: str, delay: float = 0.5) -> str:
    """GETs a single notice's detail page and pulls its description text."""
    url = NOTICE_URL_TEMPLATE.format(notice_id)
    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()
        desc = extract_description(BeautifulSoup(r.text, "html.parser"))
    except Exception as e:
        print(f"  ⚠️ Could not fetch description for notice {notice_id}: {e}")
        desc = ""
    time.sleep(delay)  # be polite — one extra request per notice
    return desc


def fetch_all_ungm_notices(page_size: int = 100, max_pages: int = 50) -> list[dict]:
    """
    Does NOT treat "got fewer results than requested" as end-of-results — every
    live payload captured from UNGM's own front-end showed a fixed page size
    regardless of what was requested, so trusting that would silently drop
    everything past page 1. Only stops on a genuinely empty page, guarded by
    max_pages so a misbehaving server can't loop forever.
    """
    session = requests.Session()
    session.headers.update(UNGM_HEADERS)
    session.get("https://www.ungm.org/Public/Notice", timeout=30)  # warm-up cookies

    raw_notices: list[dict] = []
    for page in range(max_pages):
        payload = build_payload(page_index=page, page_size=page_size)
        r = session.post(SEARCH_URL, data=json.dumps(payload), timeout=30)
        r.raise_for_status()
        batch = parse_notices(r.text)
        if not batch:
            print(f"  page {page}: empty - stopping")
            break
        raw_notices.extend(batch)
        print(f"  page {page}: {len(batch)} notices (running total {len(raw_notices)})")
        if page == max_pages - 1:
            print(f"  ⚠️ Hit max_pages={max_pages} safety cap - there may be more results. "
                  f"Raise max_pages if this happens regularly.")

    print(f"Fetching descriptions for {len(raw_notices)} notices...")
    for i, n in enumerate(raw_notices, 1):
        n["description"] = clean_description(fetch_notice_description(session, n["notice_id"]))
        if i % 10 == 0 or i == len(raw_notices):
            print(f"  ...{i}/{len(raw_notices)} done")

    return raw_notices


### Run the fetch, dedup against past uploads, build Notion-ready rows

In [3]:
raw_notices = fetch_all_ungm_notices()
print(f"\nFetched {len(raw_notices)} notices total")

# 1) Load already-uploaded titles to avoid duplicates
csv_path = "ungm_contract_titles.csv"
existing_titles = set()
if os.path.exists(csv_path):
    try:
        prev = pd.read_csv(csv_path)
        if "Title" in prev.columns:
            existing_titles = set(prev["Title"].dropna().astype(str).str.strip().str.lower())
    except Exception as e:
        print("Warning: could not read", csv_path, ":", e)

# 2) Build Notion-ready rows, skipping anything already uploaded
extracted_data = []
for n in raw_notices:
    title = (n.get("title") or "").strip()
    if not title or title.lower() in existing_titles:
        continue

    extracted_data.append({
        "closing_date": clean_ungm_deadline(n["deadline_raw"]),
        "country": n["country"] or "Not Disclosed",
        "client": n["agency"] or "Not Disclosed",
        "client_link": "",  # no stable per-agency URL available from this source
        "link": n["url"],
        "title": title,
        "description": n["description"] or "Not Disclosed",
        "value": "Not Disclosed",  # UNGM notices don't publish a contract value
        "cpv_codes": "Not Available",  # per-notice UNSPSC codes not yet parsed - see notes at top
        "language": "English",
    })

print(f"✅ {len(extracted_data)} new contracts ready for Notion upload")


  page 0: 15 notices (running total 15)


  page 1: 15 notices (running total 30)


  page 2: 15 notices (running total 45)


  page 3: 15 notices (running total 60)


  page 4: 15 notices (running total 75)


...

  ...40/79 done


  ...50/79 done


  ...60/79 done


  ...70/79 done


  ...79/79 done

Fetched 79 notices total
✅ 79 new contracts ready for Notion upload


### Upload to Notion

In [4]:
def create_page(properties: dict):
    url = "https://api.notion.com/v1/pages"
    payload = {"parent": {"database_id": DATABASE_ID}, "properties": properties}
    res = requests.post(url, headers=headers_notion, json=payload, timeout=60)
    if not res.ok:
        print("❌ Notion error:", res.status_code, res.text[:500])
    else:
        print(f"✅ Page created: {properties['Name']['title'][0]['text']['content']}")
    return res


def _safe_str(x, default=""):
    if x is None:
        return default
    s = str(x).strip()
    return s if s else default


def _safe_iso(dt_str):
    if not dt_str:
        return None
    try:
        parsed = _dateparser.parse(dt_str)
        return parsed.isoformat()
    except Exception:
        return None


now_iso = datetime.now(timezone.utc).isoformat()
new_titles_for_csv = []

# Upload newest first, matching Find_tender_notion.ipynb's convention
for contract in list(reversed(extracted_data)):
    name = _safe_str(contract.get("title"))[:1000]
    if not name:
        continue

    closing_date_iso = _safe_iso(contract.get("closing_date"))

    props = {
        "Name": {"title": [{"text": {"content": name}}]},
        "CPV Codes": {"rich_text": [{"text": {"content": _safe_str(contract.get("cpv_codes"))[:2000]}}]},
        "Client": {"rich_text": [{"text": {"content": _safe_str(contract.get("client"), "Not Disclosed")[:2000]}}]},
        "Contract Link": {"url": _safe_str(contract.get("link")) or None},
        "Date Added": {"date": {"start": now_iso, "end": None}},
        "Closing Date": {"date": {"start": closing_date_iso, "end": None}} if closing_date_iso else {"date": None},
        "Description": {"rich_text": [{"text": {"content": _safe_str(contract.get("description"), "Not Disclosed")[:2000]}}]},
        "Employer Website": {"url": _safe_str(contract.get("client_link")) or None},
        "Language": {"rich_text": [{"text": {"content": _safe_str(contract.get("language"))[:2000]}}]},
        "Location": {"rich_text": [{"text": {"content": _safe_str(contract.get("country"))[:2000]}}]},
        "Reviewed By": {"select": {"name": "N/A"}},
        "Review Status": {"select": {"name": "Not Reviewed"}},
        "Value": {"rich_text": [{"text": {"content": _safe_str(contract.get("value"), "Unavailable")[:2000]}}]},
        "Contract Status": {"select": {"name": "Open"}},
        "Source": {"select": {"name": "UNGM"}},
    }

    try:
        create_page(props)
        new_titles_for_csv.append({"Title": name})
    except Exception as e:
        print(f"Error creating Notion page for '{name}': {e}")

# Save newly uploaded titles to CSV for next run's dedup
if new_titles_for_csv:
    new_df = pd.DataFrame(new_titles_for_csv, columns=["Title"])
    header_needed = not os.path.exists(csv_path)
    new_df.to_csv(csv_path, mode="a", header=header_needed, index=False)

print(f"✅ Uploaded {len(new_titles_for_csv)} new UNGM contracts to Notion.")


✅ Page created: Provision of Services for WMO Website Hosting, Maintenance, Security, and Development


✅ Page created: REQUEST FOR PROPOSAL for Development of Four Courses within the Joint ISDB/ISFD/GPE funded Project: SMART-ED


✅ Page created: HQ26NF207_EOI for Global LTAs for Nutrition Research Services


✅ Page created: Oracle Flexcube Upgrade and Managed Services


✅ Page created: Development, Validation and Operationalization of a Regional Capacity Strengthening Learning System for Community-Based Organizations (CBOs), Refugee-Led Organizations (RLOs) and Local Actors in West


...

✅ Page created: Venue for MNCH Commodity Tracking System in Abuja for 18th to 19th Aug 2026


✅ Page created: Engineering services for the Project Design, Site Supervision and Quality Assurance of Prefabricated and Modular Infrastructure, including complementary civil works in Schools accross the West Bank


✅ Page created: Office Furniture and Equipment for Security and Justice Institutions


✅ Page created: Réunion du comité technique, suite atelier diagnostic - 05 Août 2026


✅ Page created: RFQ/206/2026/GAH/ROAS - Event management service for Social Dialogue Forum in Erbil 6 and 18 August 2026
✅ Uploaded 79 new UNGM contracts to Notion.
